# Nightingale: the deep ranker B1-DL, a small transformer (EXP-021)

The owner chose one deep model for the project, **a small transformer** (decision D-12,
2026-10-12). This notebook trains it on a Colab GPU, where the owner chose to run it (D-7). It
**replaces logistic regression in the pipeline only if it is at least as safe** (D-13), and that
is judged afterwards on the laptop, on validation, by the rule recorded in `docs/08` (EXP-021)
before this notebook was run.

| Arm | Encoder | Like |
|---|---|---|
| **B1-DL′+aug** | with the "asked" channel, 691 columns (`ddd14019c66eff10`) | B1-LR′+aug, the configured ranker |
| **B1-DL+aug** | without it, 607 columns (`3a0d5a5e01d7f427`) | B1-LR+aug; the protocol's A0 uses this arm |

Each arm is trained at the five seeds of `docs/05` §6, **42, 43, 44, 45 and 46**, on exactly the
rows its comparators used at that seed (the train split plus one masked copy of every patient),
stopping early on XGBoost's held-out 10%. Nothing is tuned (T-6). Seed 42 is the model used; the
other seeds show how much a result depends on the seed. Validate is scored at full evidence here;
the 50% and 25% levels are scored at the end (optional) and on the laptop. **The test split is
never downloaded.**

**Before you start:**
1. *Runtime → Change runtime type → T4 GPU.* Without a GPU the run would take days: stop.
2. The run saves each finished model to **your Google Drive** (`MyDrive/nightingale_b1_dl`), so if
   Colab disconnects you lose at most the arm in progress. Allow the Drive prompt. If you would
   rather not, set `USE_DRIVE = False` below; the models then live only in this session.

**Then:** *Runtime → Run all*. The smoke cell trains one epoch on 1% of the patients and prints how
long a full arm should take. The full run is roughly **2–4 hours**: about 20 minutes to download and
decode the data, then about 15–25 minutes per arm, ten arms (seed 42's two first). If Colab stops
you (its free GPU time is limited), reconnect, set the runtime to T4 again and *Run all*: finished
arms are skipped. Kaggle Notebooks work the same way if Colab will not give you a GPU.

At the end your browser downloads **`nightingale_b1_dl.zip`**: the models, their metrics and
`run.json`. No patient rows are in it. **Keep this notebook and its outputs private**: models
trained on DDXPlus are never published (`docs/11` §4). Nothing here needs a password or a token.


In [ ]:
# The commit Claude gave you. "master" works too: run.json records the exact commit either way.
COMMIT = "master"
USE_DRIVE = True

In [ ]:
import os

if not os.path.exists("/content/nightingale"):
    !git clone -q https://github.com/HarshRohila02/nightingale.git /content/nightingale
%cd /content/nightingale
!git checkout -q {COMMIT}
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: change the runtime type to T4 GPU"
import torch
print("torch", torch.__version__, "CUDA available:", torch.cuda.is_available())

In [ ]:
# Where finished models go: your Drive, so a disconnect loses nothing that is finished.
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/nightingale_b1_dl"
else:
    OUT = "/content/nightingale/models/b1_dl"
os.makedirs(OUT, exist_ok=True)
print("models go to", OUT)

In [ ]:
# DDXPlus from Hugging Face, at the snapshot the laptop uses (docs/11 §4).
# test.csv is never fetched before Phase 4 (docs/05 §2).
from huggingface_hub import hf_hub_download

for name in ["release_evidences.json", "release_conditions.json", "train.csv", "validate.csv"]:
    hf_hub_download(
        "aai530-group6/ddxplus",
        name,
        repo_type="dataset",
        revision="2ad986acc1ec62fb4a94171acc43f4fdd5bfde53",
        local_dir="data/raw/ddxplus",
    )
!ls -lh data/raw/ddxplus

In [ ]:
# Decode the vocabulary, then keep the 13 chest-pain conditions of each split (task 1a).
# Skipped if a previous run in this session already built them.
if not os.path.exists("data/interim/ddxplus_chestpain_train.parquet"):
    !python scripts/decode_ddxplus.py
    !python scripts/build_ddxplus_chestpain.py --split train
    !python scripts/build_ddxplus_chestpain.py --split validate

In [ ]:
# Smoke test: 1% of the patients, one epoch, both arms. Checks the whole job in a few minutes
# and prints how long a full arm should take. Nothing is saved.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again"
!python scripts/train_deep.py --device {DEVICE} --smoke --out /content/smoke

In [ ]:
# The real run: seeds 42-46, both arms each, seed 42 first. Finished arms are skipped.
import subprocess

for seed in (42, 43, 44, 45, 46):
    print(f"######## seed {seed} ########", flush=True)
    subprocess.run(
        ["python", "scripts/train_deep.py", "--device", DEVICE, "--seed", str(seed), "--out", OUT],
        check=True,
    )
!ls {OUT}/*

In [ ]:
# One zip to bring back: every arm's model, metrics and run.json. No patient rows are in it.
import shutil
from google.colab import files

archive = shutil.make_archive("/content/nightingale_b1_dl", "zip", OUT)
!du -sh {OUT} && ls -lh {archive}
files.download(archive)

**Optional, after the download:** score every seed at 50% and 25% of the history, on the
recorded masks (`docs/05` §3.7). The script refuses to run if the masks do not reproduce their
recorded digests. This uses the CPU only and takes about 15–30 minutes; the laptop scores seed 42
anyway. If you run it, download the second zip at the end.

In [ ]:
for seed in (42, 43, 44, 45, 46):
    result = subprocess.run(
        [
            "python", "scripts/evaluate_reduced_evidence.py", "--no-graph", "--no-red-flags",
            "--out", f"{OUT}/reduced_evidence_seed{seed}.json",
            "--model", f"B1-DL+aug={OUT}/b1_dl_aug/seed{seed}:deep",
            "--model", f"B1-DL′+aug={OUT}/b1_dl_asked_aug/seed{seed}:deep",
        ]
    )
    print(f"seed {seed}: exit code {result.returncode}")
archive = shutil.make_archive("/content/nightingale_b1_dl_with_reduced", "zip", OUT)
files.download(archive)